# Inspeção predial com visão computacional

Classificação de fissuras em concreto (Concrete Crack Images for Classification) com MobileNetV3-Small e transferência de aprendizado. O resultado alimenta a aba **Inspeção predial** do CitizenTech.

**Etapas:** 1. preparo dos dados · 2. treino · 3. avaliação · 4. desempenho.

Execute a partir da raiz do repositório, com o ambiente `.venv` ativado e as imagens extraídas em `vision/data/raw/Positive` e `vision/data/raw/Negative`.

In [ ]:
%cd ../..

## 1. Dados

A base tem recortes de 227×227 px de superfícies de concreto, com e sem fissura. Usamos uma amostra balanceada. Os nomes dos arquivos não indicam a foto de origem, então a divisão em treino, validação e teste é feita **por grupos visuais** (KMeans sobre miniaturas), de modo que superfícies parecidas não caiam em conjuntos diferentes.

In [ ]:
!python -m vision.scripts.prepareData

In [ ]:
import pandas as pd
tabela = pd.read_csv('vision/data/splits.csv')
display(tabela.groupby(['split', 'label']).size().unstack())
tabela.head()

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image
amostra = tabela[tabela['split'] == 'train'].groupby('label').head(4).sort_values('label')
figura, eixos = plt.subplots(2, 4, figsize=(10, 5))
for eixo, (indice, linha) in zip(eixos.ravel(), amostra.iterrows()):
    eixo.imshow(Image.open(linha['path']))
    eixo.set_title('com fissura' if linha['label'] == 1 else 'sem fissura')
    eixo.axis('off')

## 2. Modelo e treino

- **Arquitetura:** MobileNetV3-Small pré-treinada no ImageNet, com a última camada trocada por 2 classes.
- **Pré-processamento:** redimensiona para 224×224 e normaliza com média e desvio do ImageNet.
- **Aumento de dados:** giros, rotação, variação de brilho e contraste e desfoque leve, para tolerar sombra e iluminação.
- **Duas fases:** 1) só a cabeça de classificação, 2) ajuste fino de toda a rede com taxa menor.
- **Seleção:** a época de maior F1 na validação é exportada para ONNX.

In [ ]:
!python -m vision.scripts.trainModel

## 3. Avaliação

O limiar de confiança é escolhido na **validação** pelo maior F2, que pesa mais o recall: na triagem para inspeção é pior deixar passar uma fissura do que mandar verificar um local sem dano. O teste só mede o resultado final.

In [ ]:
!python -m vision.scripts.evaluateModel

In [ ]:
import json
from IPython.display import Image as Imagem, display
metricas = json.load(open('models/metrics.json', encoding='utf-8'))
print('Limiar recomendado:', metricas['recommendedThreshold'])
print('Teste:', {chave: metricas['test'][chave] for chave in ('precision', 'recall', 'f1', 'accuracy')})
display(pd.DataFrame(metricas['robustness']))
for nome in ('confusionMatrix', 'thresholdSweep', 'trainingCurves'):
    display(Imagem(filename=f'docs/prints/{nome}.png'))

### Acertos e erros

Exemplos de falsos positivos e falsos negativos do conjunto de teste.

In [ ]:
import os
for nome in ('falsePositives', 'falseNegatives'):
    caminho = f'docs/prints/{nome}.png'
    if os.path.exists(caminho):
        display(Imagem(filename=caminho))

## 4. Desempenho

Latência por imagem e taxa de processamento (FPS) com ONNX Runtime na CPU, registrando o equipamento e a resolução.

In [ ]:
!python -m vision.scripts.benchmarkModel

In [ ]:
!python -m vision.scripts.buildSamples

In [ ]:
!python -m vision.scripts.buildReport